In [ ]:
from pathlib import Path
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project
import pandas as pd
import os
import uuid
import shutil

In [ ]:
os.environ["KEDRO_PACKAGE_NAME"] = "altr_model"

workspace_dir = Path("workspace/results")

tags=[
    "altrisk",
    "reporting"
    ]


In [ ]:
# Since the notebook is in ./notebooks, set the project path to the parent directory
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    os.chdir(current_dir.parent)
    print(f"Changed directory from {current_dir} to {Path.cwd()}")
else:
    print(f"Already in correct directory: {current_dir}")

metadata = bootstrap_project(project_path=Path.cwd())

In [ ]:

workspace_dir.mkdir(parents=True, exist_ok=True)
print(f"Created workspace directory: {workspace_dir}")


In [ ]:
# Set companies_selection = None to cover all companies
companies_selection = [ 
    "CP_7367629456451130624",
    "CP_6411932603966179940",
    "CN_2272387057764697337",
    "CN_8289062574935601752",
    "CP_6280725841277318280",
    "CN_9147469651065703957",
    "CP_4147148409075105689",
    "CN_6475749649582456139",
    "CN_6488161088428600082",
    "CP_8560203160377002286",
    "CP_774336548921366747",
    "CP_4137086261914115",
    "CP_7605127394594156381",
    "CP_5374584170705946147",
    "CN_8031185245170948262",
    "CP_7370862721649467085",
    "CN_3371785431787292505",
    "CP_6311715612862685805",
    "CN_8064818311359297904",
    "CN_1784098929884717109",
    "CP_8927133391983718557",
    "CN_4117655119178337552",
    "CN_5490653372497379599",
    "CP_9187576119013201591",
    "CN_3609115420225854555",
    "CN_6980468522228318247",
    "CN_4440463050459774439",
    "CN_6741456596082465682",
    "CP_3185202543523296674",
    "CN_6894373266496773215",
]


# RUN

In [ ]:
# Define your parameter overrides

runs_configuration = {

    "AIM-CGE 2.2__runC_asset_granularity_with_retirement":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement_baseline": True,  
        "apply_retirement_shock": True,     
        "apply_decreasing_staggered_shock":True,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
        "apply_continued_om_baseline": False,
        "apply_continued_om_shock": False,
        "max_forecast_horizon":5
    },
    "AIM-CGE 2.2__runA_company_granularity_with_shock_continued_omcost":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": True,
        "apply_retirement_baseline": False,  
        "apply_retirement_shock": False,     
        "apply_decreasing_staggered_shock":False,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f" ,
        "apply_continued_om_baseline": False,
        "apply_continued_om_shock": True,
        "max_forecast_horizon":5
    },
    "AIM-CGE 2.2__runB_asset_granularity_with_shock_retirement":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement_baseline": False,  
        "apply_retirement_shock": True,     
        "apply_decreasing_staggered_shock":True,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
        "apply_continued_om_baseline": False,
        "apply_continued_om_shock": False,        
        "max_forecast_horizon":5
    },    
    "AIM-CGE 2.2__runD_asset_granularity_with_retirement_and_om_cost":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement_baseline": True,  
        "apply_retirement_shock": True,     
        "apply_decreasing_staggered_shock":True,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
        "apply_continued_om_baseline": True,
        "apply_continued_om_shock": True,        
        "max_forecast_horizon":5
    },        
    "AIM-CGE 2.2__runE_company_granularity_with_shock_continued_omcost":{
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": True,
        "apply_retirement_baseline": False,  
        "apply_retirement_shock": False,     
        "apply_decreasing_staggered_shock":False,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f" ,
        "apply_continued_om_baseline": False,
        "apply_continued_om_shock": True,
        "max_forecast_horizon":1
    },
    "AIM-CGE 2.2__runF_asset_granularity_with_retirement_and_om_cost":{  
        "company_ids": companies_selection,
        "reduce_granularity_from_asset_to_company_level": False,
        "apply_retirement_baseline": True, 
        "apply_retirement_shock": True,     
        "apply_decreasing_staggered_shock":True,
        "baseline_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_1200f",
        "target_scenario": "AR6_AIM/CGE 2.2_EN_NPi2020_900f",
        "apply_continued_om_baseline": True,
        "apply_continued_om_shock": True,        
        "max_forecast_horizon":1
    },        


}


In [ ]:
from IPython.display import clear_output
import sys
import traceback
from io import StringIO

all_company_trajectories = {}
all_asset_trajectories = {}
all_companies_npvs = {}
all_run_params = {}

total_runs = len(runs_configuration)

for idx, (run_name, run_params) in enumerate(runs_configuration.items(), start=1):
    clear_output(wait=True)  # clears the cell output each iteration
    
    print("================================================")
    print("================================================")
    print(f"Running {run_name}...")
    print(f"Run {idx}/{total_runs}")
    print("================================================")
    print("================================================")
    
    try:
        with KedroSession.create(
            project_path=Path.cwd(),
            extra_params=run_params,
        ) as session:
            session.run(pipeline_name="__default__", tags=tags)

            run_id = uuid.uuid4()

            company_trajectories = pd.read_csv(
                "data/07_model_output/company_trajectories.csv"
            )
            company_trajectories["run_id"] = run_id
            asset_trajectories = pd.read_csv(
                "data/07_model_output/asset_trajectories.csv"
            )
            asset_trajectories["run_id"] = run_id
            yearly_npv_trajectories = pd.read_csv(
                "data/07_model_output/yearly_npv_trajectories.csv"
            )
            yearly_npv_trajectories["run_id"] = run_id
            asset_npvs = pd.read_csv(
                "data/07_model_output/asset_npv.csv"
            )
            asset_npvs["run_id"] = run_id
            company_technology_npvs = pd.read_csv(
                "data/07_model_output/company_technology_npv.csv"
            )
            company_technology_npvs["run_id"] = run_id
            companies_npvs = pd.read_csv(
                "data/07_model_output/company_npv.csv"
            )
            companies_npvs["run_id"] = run_id

            run_params_df = pd.DataFrame([run_params])
            run_params_df["run_id"] = run_id

            all_company_trajectories[run_name] = company_trajectories
            all_asset_trajectories[run_name] = asset_trajectories
            all_companies_npvs[run_name] = companies_npvs
            all_run_params[run_name] = run_params_df

            # Copy plot folders to {workspace_dir}/{run_name}/
            run_workspace_dir = workspace_dir / run_name
            run_workspace_dir.mkdir(parents=True, exist_ok=True)

            company_trajectories.to_csv(run_workspace_dir / "company_trajectories.csv", index=False)
            asset_trajectories.to_csv(run_workspace_dir / "asset_trajectories.csv", index=False)
            asset_npvs.to_csv(run_workspace_dir / "asset_npv.csv", index=False)
            company_technology_npvs.to_csv(run_workspace_dir / "company_technology_npv.csv", index=False)
            companies_npvs.to_csv(run_workspace_dir / "company_npv.csv", index=False)
            yearly_npv_trajectories.to_csv(run_workspace_dir / "yearly_npv_trajectories.csv", index=False)
            run_params_df.to_csv(run_workspace_dir / "run_params.csv", index=False)
            
            if "reporting" in tags:
                # Copy companies_trajectories_plots
                src_trajectories = Path("data/08_reporting/companies_trajectories_plots")
                dst_trajectories = run_workspace_dir / "companies_trajectories_plots"
                if src_trajectories.exists():
                    if dst_trajectories.exists():
                        shutil.rmtree(dst_trajectories)
                    shutil.copytree(src_trajectories, dst_trajectories)
                    print(f"Copied companies_trajectories_plots to {dst_trajectories}")
                
                # Copy companies_staggered_shock_plots  
                src_staggered = Path("data/08_reporting/companies_staggered_shock_plots")
                dst_staggered = run_workspace_dir / "companies_staggered_shock_plots"
                if src_staggered.exists():
                    if dst_staggered.exists():
                        shutil.rmtree(dst_staggered)
                    shutil.copytree(src_staggered, dst_staggered)
                    print(f"Copied companies_staggered_shock_plots to {dst_staggered}")

                # Copy companies_staggered_shock_plots  
                src_staggered = Path("data/08_reporting/asset_financial_trajectories")
                dst_staggered = run_workspace_dir / "asset_financial_trajectories"
                if src_staggered.exists():
                    if dst_staggered.exists():
                        shutil.rmtree(dst_staggered)
                    shutil.copytree(src_staggered, dst_staggered)
                    print(f"Copied asset_financial_trajectories to {dst_staggered}")
                
        print(f"✅ Successfully completed {run_name}")
        
    except Exception as e:
        # Capture the error and traceback
        error_msg = f"❌ Error in {run_name}:\n"
        error_msg += f"Exception: {str(e)}\n"
        error_msg += f"Traceback:\n{traceback.format_exc()}\n"
        
        # Save error to file in the same root as the run folder
        error_file = workspace_dir / f"{run_name}_error.txt"
        with open(error_file, 'w') as f:
            f.write(error_msg)
        
        print(f"❌ Error in {run_name} - saved to {error_file}")
        print(f"Continuing with next run...")
        
        # Continue to next iteration
        continue
    else:
        print(f"{'✅'*total_runs} Successfully completed all runs")
